In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

## Change Detection with TorchGeo Tutorial
_Written by: Harald Kristen_

In this tutorial, we demonstrate how to train a change detection model using TorchGeo. 

**What is Change Detction**: 
    Change detection is the process of identifying differences between images of the same location captured at different times. It's a fundamental task in remote sensing with numerous real-world applications:

- **Urban Monitoring**: Track city growth, new construction, and infrastructure development
- **Disaster Response**: Assess damage from floods, earthquakes, wildfires, and hurricanes
- **Deforestation Tracking**: Monitor illegal logging and forest loss
- **Agriculture**: Detect crop changes, irrigation patterns, and land use shifts
- **Environmental Monitoring**: Track glacial retreat, coastal erosion, and wetland changes

In this tutorial, we'll train a binary change detection model on the [OSCD100](https://torchgeo.readthedocs.io/en/stable/api/datasets.html#oscd100) dataset to detect urban changes in bi-temporal Sentinel-2 imagery. Our model will learn to identify where new buildings have been constructed or existing ones removed.

It's recommended to run this notebook on Google Colab if you don't have your own GPU. Click the "Open in Colab" button above to get started.

## Setup

First, we install TorchGeo and TensorBoard.

In [ ]:
%pip install torchgeo tensorboard gdown

## Imports

Next, we import TorchGeo and any other libraries we need.

In [ ]:
%matplotlib inline
%load_ext tensorboard

import os
import tempfile

import matplotlib.pyplot as plt
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from torchgeo.datamodules import OSCD100DataModule
from torchgeo.datasets import OSCD100
from torchgeo.trainers import ChangeDetectionTask

## Visualize the Dataset

Let's download the dataset and look at some examples from OSCD100.

**About OSCD100**: This is a smaller version of the full [OSCD](https://torchgeo.readthedocs.io/en/stable/api/datasets.html#oscd) (Onera Satellite Change Detection) dataset for tutorials and demos. It contains 100 RGB image pairs (60 train, 20 val, 20 test) at 256×256 resolution, cropped from Sentinel-2 scenes.

OSCD100 is great for learning, but not suitable for benchmarking due to its small size. For research, use the full OSCD dataset with all 13 Sentinel-2 bands.

In [ ]:
# Load dataset
root = os.path.join(tempfile.gettempdir(), 'oscd100')
dataset = OSCD100(root=root, split='train', download=True)

# Get a sample
sample = dataset[0]

# Plot
fig = dataset.plot(sample, suptitle='OSCD100 Sample')
# Use bilinear interpolation for smoother visualization
for ax in fig.axes[2:]:
    for img in ax.get_images():
        img.set_interpolation('none')
plt.show()

print(f'Dataset size: {len(dataset)} image pairs in the {dataset.split} split')
print(f'Image shape: {sample["image"].shape} (T, C, H, W) where T=2 timesteps')
print(f'Mask shape: {sample["mask"].shape}')

This is how the data is organized on your disk by TorchGeo.

In [ ]:
dataset.directories

In [ ]:
dataset.files[:2]

## Lightning Modules

We use [Lightning](https://lightning.ai/docs/pytorch/stable/) to organize the training code and dataloader setup. The `OSCD100DataModule`:
1. Downloads the data
2. Sets up train, validation, and test dataloaders
3. Applies preprocessing and augmentation

The following variables can be modified to control training.

In [ ]:
batch_size = 16
num_workers = 4
max_epochs = 50
fast_dev_run = False

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'oscd100')
datamodule = OSCD100DataModule(
    root=root, batch_size=batch_size, num_workers=num_workers, download=True
)

We use the `ChangeDetectionTask` class from `torchgeo.trainers`, which handles:
- Concatenating the two input images into a single 6-channel tensor
- Training a U-Net model to predict change masks
- Computing metrics like accuracy, precision, recall, and F1-score



In this example, we create a `ChangeDetectionTask` object with a U-Net model using a ResNet-18 backbone. We use ImageNet pretrained weights (`weights=True`), which helps the model learn faster and achieve better performance than training from scratch.  
For other supported model and backbones, check the [trainers documentation](https://torchgeo.readthedocs.io/en/stable/api/trainers.html#torchgeo.trainers.ChangeDetectionTask) for more information.

In [ ]:
task = ChangeDetectionTask(
    model='unet',
    backbone='resnet18',
    weights=True,  # ImageNet pretrained weights
    loss='bce',
    in_channels=3,  # RGB
    lr=0.001,
)

## Training

Now we can train the model using Lightning's [Trainer](https://lightning.ai/docs/pytorch/stable/common/trainer.html), that allows us to easily do:

- Model checkpointing, that saves the best model based on a metric we define
- Early stopping to prevent overfitting
- TensorBoard logs metrics for visualization

In [ ]:
default_root_dir = os.path.join(tempfile.gettempdir(), 'experiments')
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss', dirpath=default_root_dir, save_top_k=1, save_last=True
)
early_stopping_callback = EarlyStopping(monitor='val_loss', min_delta=0.0, patience=10)
logger = TensorBoardLogger(save_dir=default_root_dir, name='change_detection_logs')

In [ ]:
trainer = Trainer(
    callbacks=[checkpoint_callback, early_stopping_callback],
    log_every_n_steps=1,
    logger=logger,
    min_epochs=1,
    max_epochs=max_epochs,
)

**Note** Training runs for ~1 minute, occupying 3 GB of VRAM until max epochs is reached.

In [ ]:
trainer.fit(model=task, datamodule=datamodule)

## Visualize Training with TensorBoard

Use TensorBoard to visualize metrics across epochs.

**In Google Colab**, run the cell below to launch TensorBoard inline.

**Locally**, run from your terminal `tensorboard --logdir /tmp/experiments` and navigate to http://localhost:6006

In [ ]:
%tensorboard --logdir "$default_root_dir"

## Evaluation

Now we evaluate the model on the test set.

### Understanding the Metrics

- **Accuracy**: Percentage of correctly classified pixels
- **F1 Score**: Harmonic mean of precision and recall
- **Jaccard Index (IoU)**: Intersection over Union between prediction and ground truth

In [ ]:
trainer.test(model=task, datamodule=datamodule, ckpt_path='best')

## Visualize Predictions

Let's visualize predictions on the test set using our best model.

In [ ]:
test_loader = datamodule.test_dataloader()
predictions = trainer.predict(task, dataloaders=test_loader, ckpt_path='best')

# Flatten predictions from batches to individual samples
all_preds = torch.cat(predictions, dim=0)

# Visualize first 3 samples
for idx in range(3):
    sample = datamodule.test_dataset[idx]

    pred_probs = all_preds[idx].cpu().squeeze()
    pred_mask = (pred_probs > 0.5).float()

    print(
        f'Sample {idx}: Prediction range [{pred_probs.min():.4f}, {pred_probs.max():.4f}], '
        f'Change pixels: {pred_mask.sum().item():.0f}/{pred_mask.numel()} '
        f'({100 * pred_mask.sum().item() / pred_mask.numel():.1f}%)'
    )

    sample['prediction'] = pred_mask.unsqueeze(0)
    fig = datamodule.test_dataset.plot(sample, suptitle=f'Test Sample {idx}')

plt.show()

### Interpreting the Results

The model achieves **72.7% F1** and **94.1% accuracy** on the test set, which is ok for a tutorial but if you want a good model you can improve results when you:

- Use the full [OSCD](https://torchgeo.readthedocs.io/en/stable/api/datasets.html#oscd) dataset with all 13 bands
- Train for 100+ epochs
- Use weighted/focal loss for class imbalance
- Tune hyperparameters
- Try different models and backbones (see options above)